# explanatory data analysis

# imports

In [46]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = duckdb.connect('../../database/financial_data.duckdb', read_only=True)

In [47]:
def query(sql):
    return conn.execute(sql).df()

# rows count 4h only

In [48]:
query("""
SELECT asset_symbol, COUNT(*) as rows, 
       MIN(date) as first_date, MAX(date) as last_date
FROM gold_crypto_features 
WHERE interval = '4h'
GROUP BY asset_symbol 
ORDER BY rows DESC
""")

,asset_symbol,rows,first_date,last_date
0,BTC,13274,2020-04-27 12:00:00,2026-05-18 16:00:00
1,LTC,12014,2020-11-23 12:00:00,2026-05-18 16:00:00
2,ETH,11146,2021-04-17 04:00:00,2026-05-18 16:00:00
3,ADA,11127,2021-04-20 08:00:00,2026-05-18 16:00:00
4,DOT,11126,2021-04-20 12:00:00,2026-05-18 16:00:00
5,XRP,10790,2021-06-15 12:00:00,2026-05-18 16:00:00
6,DOGE,10670,2021-07-05 12:00:00,2026-05-18 16:00:00
7,AVAX,10040,2021-10-18 12:00:00,2026-05-18 16:00:00
8,DYDX,9885,2021-11-13 08:00:00,2026-05-18 16:00:00
9,SOL,9862,2021-11-17 04:00:00,2026-05-18 16:00:00


# all intervals comparison

In [49]:
query("""
SELECT interval, COUNT(*) as total_rows, 
       COUNT(DISTINCT asset_symbol) as symbols
FROM gold_crypto_features 
GROUP BY interval 
ORDER BY interval
""")

,interval,total_rows,symbols
0,1d,17581,11
1,1h,472135,11
2,4h,116404,11
3,W,687,10


# nan check like oi and funding rate for 4h

In [50]:
query("""
SELECT asset_symbol,
       COUNT(*) as total_rows,
       SUM(CASE WHEN open_interest IS NULL THEN 1 ELSE 0 END) as null_oi,
       SUM(CASE WHEN funding_rate IS NULL THEN 1 ELSE 0 END) as null_fr,
       SUM(CASE WHEN rsi_14 IS NULL THEN 1 ELSE 0 END) as null_rsi,
       SUM(CASE WHEN macd IS NULL THEN 1 ELSE 0 END) as null_macd
FROM gold_crypto_features
WHERE interval = '4h'
GROUP BY asset_symbol
ORDER BY total_rows DESC
""")

,asset_symbol,total_rows,null_oi,null_fr,null_rsi,null_macd
0,BTC,13274,13274.0,12875.0,0.0,0.0
1,LTC,12014,12014.0,11615.0,0.0,0.0
2,ETH,11146,11146.0,10747.0,0.0,0.0
3,ADA,11127,11127.0,10728.0,0.0,0.0
4,DOT,11126,11126.0,10727.0,0.0,0.0
5,XRP,10790,10790.0,10391.0,0.0,0.0
6,DOGE,10670,10670.0,10271.0,0.0,0.0
7,AVAX,10040,10040.0,9641.0,0.0,0.0
8,DYDX,9885,9885.0,9486.0,0.0,0.0
9,SOL,9862,9862.0,9463.0,0.0,0.0


In [51]:
query("""
SELECT interval, COUNT(*) as rows
FROM gold_crypto_features
GROUP BY interval
ORDER BY interval
""")


,interval,rows
0,1d,17581
1,1h,472135
2,4h,116404
3,W,687


# btc 4h price stats

In [52]:
query("""
SELECT COUNT(*) as rows, ROUND(MIN(close), 2) as min_close,
       ROUND(MAX(close), 2) as max_close, ROUND(AVG(close), 2) as avg_close,
       ROUND(STDDEV(close), 2) as std_close, ROUND(AVG(volume), 0) as avg_volume
FROM gold_crypto_features
WHERE asset_symbol = 'BTC' AND interval = '4h'
""")

,rows,min_close,max_close,avg_close,std_close,avg_volume
0,13274,7669.5,125329.4,51049.62,30637.64,15757.0


# BTC 1h vs 4h comparison

In [53]:
query("""
SELECT interval, COUNT(*) as rows, MIN(date) as first_date, MAX(date) as last_date
FROM gold_crypto_features
WHERE asset_symbol = 'BTC' AND interval IN ('1h', '4h')
GROUP BY interval
""")

,interval,rows,first_date,last_date
0,4h,13274,2020-04-27 12:00:00,2026-05-18 16:00:00
1,1h,53688,2020-04-02 17:00:00,2026-05-18 16:00:00


In [54]:
conn.close()